# K-Fold v5.1 Comparative Full (Train + Compare in One Notebook)

This notebook trains both models from scratch with super-small defaults (`n_splits=1`, `epochs=1`) and then compares them on the same holdout fold.

No checkpoint recovery is used: each training run starts fresh and writes new `model_fold0.pt` files.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "v5_compartive"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("Git branch:", current_branch)
print("Git commit:", current_commit)
print("Repo ready at:", REPO_DIR)


In [ ]:
import os
import gc
import json
import random
import argparse
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import (
    TweetExtractionCausalDataset,
    causal_collate_fn,
    register_special_tokens,
    normalize_kaggle_span_text,
    normalize_text,
    SPECIAL_TOKENS,
)
from dz2_causal.losses import compute_total_loss
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard
from dz2_causal.compare_v3_v4_jaccard import run_comparison

V3_CFG_PATH = os.environ.get("V3_CFG_PATH", "config/kaggle_train_kfold_v3_parallel_t4x2_super_small.json")
V4_CFG_PATH = os.environ.get("V4_CFG_PATH", "config/kaggle_train_kfold_v4_jaccard_t4x2_super_small.json")
COMPARE_CFG_PATH = os.environ.get("COMPARE_CFG_PATH", "config/kaggle_compare_v3_v4_jaccard_super_small.json")
RUN_NAME = os.environ.get("RUN_NAME", "v5_1_compartive_full_super_small")

print("Using V3 config:", V3_CFG_PATH)
print("Using V4 config:", V4_CFG_PATH)
print("Using compare config:", COMPARE_CFG_PATH)

v3_cfg = json.loads(Path(V3_CFG_PATH).read_text())
v4_cfg = json.loads(Path(V4_CFG_PATH).read_text())
compare_cfg = json.loads(Path(COMPARE_CFG_PATH).read_text())

output_root = Path("/kaggle/working/twitter_sentiment_outputs") / RUN_NAME
v3_cfg["output_dir"] = str(output_root / "v3_train")
v4_cfg["output_dir"] = str(output_root / "v4_train")
compare_cfg["output_dir"] = str(output_root / "comparison")

for cfg in (v3_cfg, v4_cfg):
    cfg["n_splits"] = 1
    cfg["epochs"] = 1
    cfg["single_fold_val_fraction"] = float(cfg.get("single_fold_val_fraction", 0.2))

output_root.mkdir(parents=True, exist_ok=True)
(output_root / "effective_configs.json").write_text(
    json.dumps({"v3": v3_cfg, "v4": v4_cfg, "compare": compare_cfg}, indent=2)
)

print("Output root:", output_root)
print("No checkpoint recovery: enabled by design (fresh training each run).")
{
    "v3_output": v3_cfg["output_dir"],
    "v4_output": v4_cfg["output_dir"],
    "comparison_output": compare_cfg["output_dir"],
}


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_records(frame: pd.DataFrame, prompt_text: str):
    records = frame[["text", "sentiment", "selected_text"]].to_dict("records")
    for r in records:
        r["prompt"] = prompt_text
    return records


def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def align_head_dtypes_with_lm(model, device):
    base_model = unwrap_model(model)
    target_dtype = base_model.lm.get_input_embeddings().weight.dtype
    base_model.start_head.to(device=device, dtype=target_dtype)
    base_model.end_head.to(device=device, dtype=target_dtype)
    base_model.select_head.to(device=device, dtype=target_dtype)
    return target_dtype


def reduce_loss_tensor(loss_val: torch.Tensor) -> torch.Tensor:
    if torch.is_tensor(loss_val) and loss_val.ndim > 0:
        return loss_val.mean()
    return loss_val


def loss_to_float(loss_val: torch.Tensor) -> float:
    return float(reduce_loss_tensor(loss_val).detach().item())


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()


def resolve_parallel_devices(cfg):
    if not torch.cuda.is_available():
        return []

    available_ids = list(range(torch.cuda.device_count()))
    requested_ids = cfg.get("parallel_gpu_ids", available_ids)
    requested_ids = [int(i) for i in requested_ids if int(i) in available_ids]
    if not requested_ids:
        requested_ids = available_ids

    if cfg.get("prefer_most_free_gpu", True):
        requested_ids = sorted(
            requested_ids,
            key=lambda did: torch.cuda.mem_get_info(did)[0],
            reverse=True,
        )

    if cfg.get("use_data_parallel", False) and len(requested_ids) >= 2:
        return requested_ids
    return [requested_ids[0]]


def resolve_amp_settings(cfg):
    use_amp = bool(cfg.get("use_amp", not bool(cfg.get("no_amp", False))))
    if not torch.cuda.is_available() or not use_amp:
        return False, None, False

    amp_pref = str(cfg.get("amp_dtype", "fp16")).lower()
    if amp_pref == "bf16":
        if not torch.cuda.is_bf16_supported():
            raise ValueError("amp_dtype='bf16' requested but GPU does not support bf16.")
        amp_dtype = torch.bfloat16
    elif amp_pref == "fp16":
        amp_dtype = torch.float16
    elif amp_pref == "auto":
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        raise ValueError("amp_dtype must be one of: fp16, bf16, auto")

    use_grad_scaler = amp_dtype == torch.float16
    return True, amp_dtype, use_grad_scaler


def run_smoke_step(model, loader, optimizer, scheduler, device, use_amp=False, amp_dtype=None, use_grad_scaler=False, scaler=None):
    smoke_batch = next(iter(loader))
    for k, v in smoke_batch.items():
        if torch.is_tensor(v):
            smoke_batch[k] = v.to(device)

    amp_context = (
        torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)
        if device.type == "cuda"
        else nullcontext()
    )

    with amp_context:
        out = model(
            input_ids=smoke_batch["input_ids"],
            attention_mask=smoke_batch["attention_mask"],
            labels=smoke_batch["labels"],
            compute_span_logits=False,
        )
        losses = compute_total_loss(
            outputs=out,
            batch=smoke_batch,
            lambda_kl=0.0,
            lambda_select=0.0,
        )
        loss = reduce_loss_tensor(losses["loss"])

    print(f"[smoke] loss={loss.item():.4f}")

    if use_grad_scaler:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)
    scheduler.step()


def annotate_seq_lens(df_in: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    probe = AutoTokenizer.from_pretrained(
        cfg["model_name"],
        use_fast=True,
        trust_remote_code=cfg.get("trust_remote_code", False),
    )
    register_special_tokens(probe)

    extra_tokens = int(probe.bos_token_id is not None) + int(probe.eos_token_id is not None)
    seq_lens = []

    for row in df_in[["text", "sentiment", "selected_text"]].to_dict("records"):
        prompt = cfg["prompt_text"]
        tweet = normalize_kaggle_span_text(row["text"])
        sentiment = normalize_text(row["sentiment"])
        selected_text = normalize_kaggle_span_text(row["selected_text"])

        prefix_text = (
            f"{SPECIAL_TOKENS.prompt_open} {prompt} {SPECIAL_TOKENS.prompt_close} "
            f"{SPECIAL_TOKENS.tweet_open}{tweet} {SPECIAL_TOKENS.tweet_close} "
            f"{SPECIAL_TOKENS.sentiment_open} {sentiment} {SPECIAL_TOKENS.sentiment_close} "
            f"{SPECIAL_TOKENS.answer_open}"
        )
        full_text = f"{prefix_text}{selected_text} {SPECIAL_TOKENS.answer_close}"
        tokenized = probe(full_text, add_special_tokens=False)["input_ids"]
        seq_lens.append(len(tokenized) + extra_tokens)

    out = df_in.copy()
    out["seq_len"] = seq_lens
    return out


def train_single_fold(model_label: str, cfg: dict, df_train_all: pd.DataFrame, train_idx, val_idx):
    print(f"\n===== Training {model_label} | n_splits=1 | epochs={cfg['epochs']} =====")
    print("No checkpoint recovery: starting from scratch.")

    output_dir = Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    df_local = annotate_seq_lens(df_train_all, cfg)

    train_frame = df_local.iloc[train_idx].reset_index(drop=True)
    val_frame = df_local.iloc[val_idx].reset_index(drop=True)

    if cfg.get("filter_long_samples", False):
        before = len(train_frame)
        train_frame = train_frame[train_frame["seq_len"] <= int(cfg["max_len"])].reset_index(drop=True)
        dropped = before - len(train_frame)
        print(f"[{model_label}] dropped {dropped} train samples over max_len={cfg['max_len']}")

    if len(train_frame) == 0:
        raise ValueError(f"{model_label}: no train rows left after filtering. Increase max_len.")

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["model_name"],
        use_fast=True,
        trust_remote_code=cfg.get("trust_remote_code", False),
    )

    base_model = CausalExtractionModel(
        model_name=cfg["model_name"],
        trust_remote_code=cfg.get("trust_remote_code", False),
    )
    if cfg.get("gradient_checkpointing", False):
        base_model.lm.gradient_checkpointing_enable()
        if hasattr(base_model.lm, "config"):
            base_model.lm.config.use_cache = False

    register_special_tokens(tokenizer, model=base_model.lm)

    train_ds = TweetExtractionCausalDataset(
        records=make_records(train_frame, cfg["prompt_text"]),
        tokenizer=tokenizer,
        prompt_text=cfg["prompt_text"],
        max_len=int(cfg["max_len"]),
        soft_alpha=float(cfg["soft_alpha"]),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=True,
        num_workers=int(cfg.get("num_workers", 0)),
        collate_fn=lambda b: causal_collate_fn(b, tokenizer.pad_token_id),
    )

    if len(train_loader) == 0:
        raise ValueError(f"{model_label}: empty train loader.")

    device_ids = resolve_parallel_devices(cfg)
    device = torch.device(f"cuda:{device_ids[0]}" if torch.cuda.is_available() else "cpu")

    base_model.to(device)
    model = base_model
    if torch.cuda.is_available() and bool(cfg.get("use_data_parallel", False)) and len(device_ids) > 1:
        model = torch.nn.DataParallel(base_model, device_ids=device_ids, output_device=device_ids[0])
        print(f"[{model_label}] DataParallel enabled on GPUs {device_ids}")
    else:
        print(f"[{model_label}] DataParallel disabled | device={device}")

    lm_dtype = align_head_dtypes_with_lm(base_model, device)
    use_amp, amp_dtype, use_grad_scaler = resolve_amp_settings(cfg)
    if use_grad_scaler and lm_dtype == torch.bfloat16:
        use_grad_scaler = False

    scaler = torch.amp.GradScaler("cuda", enabled=(use_grad_scaler and device.type == "cuda"))

    optimizer = torch.optim.AdamW(
        base_model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    total_steps = max(1, int(cfg["epochs"]) * len(train_loader))
    warmup_steps = int(float(cfg["warmup_ratio"]) * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    optimizer.zero_grad(set_to_none=True)

    if bool(cfg.get("run_smoke_test", True)):
        run_smoke_step(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            use_grad_scaler=use_grad_scaler,
            scaler=scaler,
        )

    model.train()
    history_rows = []
    global_step = 0

    for epoch in range(int(cfg["epochs"])):
        ce_only = epoch < int(cfg["ce_only_epochs"])
        lambda_kl = 0.0 if ce_only else float(cfg["lambda_kl"])
        lambda_select = 0.0 if ce_only else float(cfg["lambda_select"])
        need_span = (lambda_kl > 0.0) or (lambda_select > 0.0)

        running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}
        grad_accum_steps = int(cfg["grad_accum_steps"])

        for step, batch in enumerate(train_loader):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)

            amp_context = (
                torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)
                if device.type == "cuda"
                else nullcontext()
            )

            with amp_context:
                out = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                    compute_span_logits=need_span,
                )
                losses = compute_total_loss(
                    outputs=out,
                    batch=batch,
                    lambda_kl=lambda_kl,
                    lambda_select=lambda_select,
                )
                full_loss = reduce_loss_tensor(losses["loss"])
                loss = full_loss / grad_accum_steps

            if use_grad_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % grad_accum_steps == 0:
                if use_grad_scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

            running["loss"] += loss_to_float(losses["loss"])
            running["ce"] += loss_to_float(losses["ce_loss"])
            running["kl"] += loss_to_float(losses["kl_loss"])
            running["sel"] += loss_to_float(losses["select_loss"])

            if (step + 1) % int(cfg["log_every"]) == 0:
                denom = float(cfg["log_every"])
                row = {
                    "model": model_label,
                    "epoch": int(epoch + 1),
                    "step": int(step + 1),
                    "global_step": int(global_step),
                    "loss": float(running["loss"] / denom),
                    "ce": float(running["ce"] / denom),
                    "kl": float(running["kl"] / denom),
                    "sel": float(running["sel"] / denom),
                }
                history_rows.append(row)
                print(
                    f"[{model_label}] epoch={row['epoch']} step={row['step']}/{len(train_loader)} "
                    f"loss={row['loss']:.4f} ce={row['ce']:.4f} kl={row['kl']:.4f} sel={row['sel']:.4f}"
                )
                running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}

        if len(train_loader) % int(cfg["grad_accum_steps"]) != 0:
            if use_grad_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

    val_eval = evaluate_dataframe_jaccard(
        df=val_frame[["text", "sentiment", "selected_text"]].copy(),
        model=base_model,
        tokenizer=tokenizer,
        prompt_text=cfg["prompt_text"],
        device=device,
        max_new_tokens=int(cfg.get("max_new_tokens", 16)),
        constrain_to_tweet_span=bool(cfg.get("constrain_to_tweet_span", True)),
    )

    ckpt_path = output_dir / "model_fold0.pt"
    torch.save(
        {
            "model_state_dict": base_model.state_dict(),
            "model_name": cfg["model_name"],
            "fold": 0,
            "val_jaccard": float(val_eval["mean_jaccard"]),
            "config": cfg,
            "global_step": int(global_step),
        },
        ckpt_path,
    )

    history_df = pd.DataFrame(history_rows)
    history_path = output_dir / "training_history.csv"
    history_df.to_csv(history_path, index=False)

    metrics = {
        "model": model_label,
        "checkpoint": str(ckpt_path),
        "val_rows": int(len(val_frame)),
        "val_jaccard": float(val_eval["mean_jaccard"]),
        "global_step": int(global_step),
        "train_rows_after_filter": int(len(train_frame)),
    }
    metrics_path = output_dir / "training_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))

    print(
        f"[{model_label}] val_jaccard={metrics['val_jaccard']:.4f} "
        f"| train_rows={metrics['train_rows_after_filter']} "
        f"| ckpt={ckpt_path}"
    )

    del train_loader, train_ds, optimizer, scheduler, model, base_model, tokenizer
    clear_cuda_memory()

    return metrics


In [ ]:
set_seed(int(v3_cfg["seed"]))

df_full = pd.read_csv(v3_cfg["train_csv"]).dropna(subset=["text", "sentiment", "selected_text"]).reset_index(drop=True)
print("Train rows (full):", len(df_full))

train_subset_fraction = float(v3_cfg.get("train_subset_fraction", 1.0))
v4_subset_fraction = float(v4_cfg.get("train_subset_fraction", train_subset_fraction))
if abs(train_subset_fraction - v4_subset_fraction) > 1e-9:
    print("Warning: v3 and v4 train_subset_fraction differ; using v3 value for shared split.")

if not (0.0 < train_subset_fraction <= 1.0):
    raise ValueError("train_subset_fraction must be in the interval (0, 1].")

if train_subset_fraction < 1.0:
    subset_seed = int(v3_cfg.get("train_subset_seed", v3_cfg["seed"]))
    min_rows_per_class = max(2, int(v3_cfg.get("min_subset_rows_per_class", 2)))
    subset_parts = []
    for sentiment, sentiment_df in df_full.groupby("sentiment", sort=False):
        target_n = int(round(len(sentiment_df) * train_subset_fraction))
        target_n = max(min_rows_per_class, target_n)
        target_n = min(len(sentiment_df), target_n)
        subset_parts.append(sentiment_df.sample(n=target_n, random_state=subset_seed))

    df = (
        pd.concat(subset_parts, axis=0)
        .sample(frac=1.0, random_state=subset_seed)
        .reset_index(drop=True)
    )
    print(
        f"Train subset enabled: fraction={train_subset_fraction:.3f} "
        f"rows={len(df)}/{len(df_full)}"
    )
else:
    df = df_full
    print("Train subset disabled: using full dataset.")

sent_counts = df["sentiment"].value_counts()
if int(sent_counts.min()) < 2:
    raise ValueError("Need at least 2 rows per sentiment for single-fold train/val split.")

val_fraction = float(v3_cfg.get("single_fold_val_fraction", 0.2))
if not (0.0 < val_fraction < 1.0):
    raise ValueError("single_fold_val_fraction must be in the interval (0, 1).")

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=val_fraction,
    random_state=int(v3_cfg["seed"]),
)
train_idx, val_idx = next(splitter.split(df, df["sentiment"]))

split_info = {
    "rows_total": int(len(df)),
    "rows_train": int(len(train_idx)),
    "rows_val": int(len(val_idx)),
    "val_fraction": float(val_fraction),
}
(output_root / "split_info.json").write_text(json.dumps(split_info, indent=2))
print("Split info:", split_info)

eval_df = df.iloc[val_idx].reset_index(drop=True)[["text", "sentiment", "selected_text"]]
eval_csv_path = output_root / "eval_fold0.csv"
eval_df.to_csv(eval_csv_path, index=False)
print("Shared eval CSV:", eval_csv_path)


In [ ]:
v3_metrics = train_single_fold("v3_ce", v3_cfg, df, train_idx, val_idx)
v4_metrics = train_single_fold("v4_kl_jaccard", v4_cfg, df, train_idx, val_idx)

train_summary = {
    "v3": v3_metrics,
    "v4": v4_metrics,
}
(output_root / "train_summary.json").write_text(json.dumps(train_summary, indent=2))
pd.DataFrame([v3_metrics, v4_metrics])


In [ ]:
compare_output_dir = Path(compare_cfg["output_dir"])
compare_output_dir.mkdir(parents=True, exist_ok=True)

compare_args = argparse.Namespace(
    eval_csv=str(eval_csv_path),
    output_dir=str(compare_output_dir),
    model_name=compare_cfg.get("model_name", v3_cfg["model_name"]),
    prompt_text=compare_cfg.get("prompt_text", v3_cfg["prompt_text"]),
    v3_checkpoint=str(v3_metrics["checkpoint"]),
    v4_checkpoint=str(v4_metrics["checkpoint"]),
    trust_remote_code=bool(compare_cfg.get("trust_remote_code", False)),
    max_new_tokens=int(compare_cfg.get("max_new_tokens", 16)),
    constrain_to_tweet_span=bool(compare_cfg.get("constrain_to_tweet_span", True)),
    eval_subset_fraction=1.0,
    max_eval_rows=0,
    device=(compare_cfg.get("device", "cuda:0") if torch.cuda.is_available() else "cpu"),
    seed=int(compare_cfg.get("seed", 42)),
)

print("Comparison eval CSV:", compare_args.eval_csv)
print("Comparison checkpoints:")
print(" - v3:", compare_args.v3_checkpoint)
print(" - v4:", compare_args.v4_checkpoint)

comparison_results = run_comparison(compare_args)
comparison_results["summary"]


In [ ]:
comparison_dir = Path(compare_cfg["output_dir"])
summary_path = comparison_dir / "comparison_summary.json"
preds_path = comparison_dir / "v3_vs_v4_predictions.csv"

if summary_path.exists():
    print("Summary path:", summary_path)
    print(summary_path.read_text())

if preds_path.exists():
    preds_df = pd.read_csv(preds_path)
    print("Predictions rows:", len(preds_df))
    preds_df.head()
